# 02 — Structure selection by CA-RMSD

Cluster a small set of NUDT15 PDBs by CA-RMSD after superposition. The rest of
the example still uses **5lpg**, even if it sits in the same cluster as the
others.


## PDB list


In [ ]:
from IPython.display import display
import pandas as pd

from dualde.mapping import load_structural_mapping_json
from dualde.paths import example_dir
from dualde.rmsd import (
    plot_rmsd_cluster_counts,
    plot_rmsd_heatmap,
    rmsd_heatmap_after_alignment,
    summarize_rmsd_clusters_at_cutoff,
)

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
PDB_DIR = EXAMPLE / "pdb"

hits = pd.DataFrame(load_structural_mapping_json(EXAMPLE / "mapping" / "pdb_hits.json"))
print("all mapped hits:", hits.shape)
display(hits.head(10))
print("unique pdb_id:", sorted(hits["pdb_id"].unique()))


Shipped locally (offline): `5lpg`, `5bon`, `6t5j`, `7b66`. Fetch more from `hits` if you want a larger matrix.


In [ ]:
EXAMPLE_STRUCTURE = "5lpg"
local_ids = ["5lpg", "5bon", "6t5j", "7b66"]
pdb_files, labels, chain_specs = [], [], {}
for pdb_id in local_ids:
    path = PDB_DIR / f"{pdb_id}.pdb"
    print(pdb_id, "OK" if path.is_file() else "MISSING", path)
    if not path.is_file():
        continue
    pdb_files.append(str(path))
    labels.append(pdb_id)
    chain_specs[str(path)] = "A"

rmsd_df, cleaned = rmsd_heatmap_after_alignment(
    pdb_files=pdb_files,
    chain_specs=chain_specs,
    output_dir=str(PDB_DIR),
    labels=labels,
    show=False,
)
rmsd_df.to_csv(OUT / "rmsd_matrix.csv")
print("cleaned files:", cleaned)


## RMSD matrix


In [ ]:
print(rmsd_df.round(3))
print("min off-diagonal Å:", float(rmsd_df.where(~(rmsd_df.index.to_series() == rmsd_df.columns)).min().min()))


These four NUDT15 structures are nearly the same (sub-Å). Keep **5lpg** as the example PDB.


In [ ]:
CUTOFF = 2.0
summary = summarize_rmsd_clusters_at_cutoff(rmsd_df, cutoff=CUTOFF)
summary.to_csv(OUT / "rmsd_clusters.csv", index=False)
print("example structure:", EXAMPLE_STRUCTURE)
display(summary)

plot_rmsd_heatmap(rmsd_df, highlight_pdb=EXAMPLE_STRUCTURE)
plot_rmsd_cluster_counts(rmsd_df, cutoff=CUTOFF)
